# 어텐션 직관

모델은 다음 토큰을 예측할 때 입력의 어느 부분을 관련 있다고 볼까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 교재용 가중치 읽기

**입력**은 설명을 위해 토큰이라고 가정한 문자열 목록과 같은 길이의 비중 목록입니다. `zip()`은 같은 위치의 토큰과 비중을 한 쌍씩 반환합니다. `{token:12}`는 토큰을 12칸에 맞추고 `{weight:.2f}`는 비중을 소수점 둘째 자리까지 표시합니다. `"■" * int(weight * 20)`은 비중을 막대 칸 수로 바꿉니다. `sum()`은 비중의 합을 반환합니다. 이 값은 실제 모델에서 추출한 어텐션이 아니라 가중치를 읽는 법을 익히는 교재용 예입니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
tokens = ["상품", "A는", "파란색이고", "상품", "B는", "빨간색입니다."]
weights_for_b = [0.05, 0.05, 0.05, 0.10, 0.30, 0.45]
print("질문: 상품 B의 색은?")
for token, weight in zip(tokens, weights_for_b):
    print(f"{token:12} {weight:.2f} " + "■" * int(weight * 20))
print("합계:", sum(weights_for_b))

질문: 상품 B의 색은?
상품           0.05 ■
A는           0.05 ■
파란색이고        0.05 ■
상품           0.10 ■■
B는           0.30 ■■■■■■
빨간색입니다.      0.45 ■■■■■■■■■
합계: 1.0


## 질문이 바뀌면 비중도 바뀝니다

문장 토큰은 그대로 두고 질문 대상을 B에서 A로 바꾼 상황을 나타냅니다. 처리 조건이 바뀌면 큰 비중의 위치도 바뀐다는 점을 두 열에서 비교합니다. 이 셀의 화면 출력으로 실제 Qwen의 답이나 내부 어텐션을 설명할 수는 없습니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
weights_for_a = [0.10, 0.30, 0.45, 0.05, 0.05, 0.05]
for token, before, after in zip(tokens, weights_for_b, weights_for_a):
    print(f"{token:12} B질문={before:.2f} A질문={after:.2f}")
print("이 비교는 개념 설명용이며 실제 모델을 측정하지 않았습니다.")

상품           B질문=0.05 A질문=0.10
A는           B질문=0.05 A질문=0.30
파란색이고        B질문=0.05 A질문=0.45
상품           B질문=0.10 A질문=0.05
B는           B질문=0.30 A질문=0.05
빨간색입니다.      B질문=0.45 A질문=0.05
이 비교는 개념 설명용이며 실제 모델을 측정하지 않았습니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [3]:
assert abs(sum(weights_for_a) - 1.0) < 1e-9 and abs(sum(weights_for_b) - 1.0) < 1e-9
assert weights_for_b.index(max(weights_for_b)) == 5
assert weights_for_a.index(max(weights_for_a)) == 2
print("검증: 질문 대상에 따라 큰 교재용 비중의 위치가 이동했습니다.")

검증: 질문 대상에 따라 큰 교재용 비중의 위치가 이동했습니다.
